In [1]:
from copy import deepcopy
from typing import Any

from pydantic import BaseModel, Field, ConfigDict
from pydantic.json_schema import GenerateJsonSchema, JsonSchemaMode, JsonSchemaValue, SkipJsonSchema
from pydantic_core.core_schema import CoreSchema

# model stuff

In [2]:
class Foo(BaseModel):
    model_config = ConfigDict(json_schema_serialization_defaults_required=True)

    a: str
    b: str = "b"
    c: str | None = "c"
    d: str = Field("d", json_schema_extra=lambda x: x.pop("default"))
    e: SkipJsonSchema[str] | None = "e"
    f: str | SkipJsonSchema[None] = "f"
    g: str | SkipJsonSchema[None] = Field("g", json_schema_extra=lambda x: x.pop("default"))
    h: str | SkipJsonSchema[None]

In [3]:
Foo.model_json_schema(mode="serialization")

{'properties': {'a': {'title': 'A', 'type': 'string'},
  'b': {'default': 'b', 'title': 'B', 'type': 'string'},
  'c': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': 'c',
   'title': 'C'},
  'd': {'title': 'D', 'type': 'string'},
  'f': {'default': 'f', 'title': 'F', 'type': 'string'},
  'g': {'title': 'G', 'type': 'string'},
  'h': {'title': 'H', 'type': 'string'}},
 'required': ['a', 'b', 'c', 'd', 'f', 'g', 'h'],
 'title': 'Foo',
 'type': 'object'}

# custom schema serialization

In [4]:
def recursively_remove_examples(data: JsonSchemaValue) -> None:
    if isinstance(data, dict):
        data.pop("examples", None)
        for value in data.values():
            recursively_remove_examples(value)
    elif isinstance(data, list):
        for item in data:
            recursively_remove_examples(item)

class SchemaWithoutExamples(GenerateJsonSchema):
    def generate(self, schema: CoreSchema, mode: JsonSchemaMode = "validation") -> JsonSchemaValue:
        json_schema = super().generate(schema, mode=mode)
        recursively_remove_examples(json_schema)
        return json_schema

In [5]:
class InlineSchema(GenerateJsonSchema):
    def generate(self, schema: CoreSchema, mode: JsonSchemaMode = "validation") -> JsonSchemaValue:
        json_schema = super().generate(schema, mode=mode)

        if "$defs" not in json_schema:
            return json_schema

        defs = json_schema.pop("$defs")

        def replace_refs(obj: Any) -> Any:
            if isintance(obj, dict):
                if "$ref" in obj:
                    ref_path = obj["$ref"]
                    def_key = ref_path.split("/")[-1]
                    definition = deepcopy(defs.get(def_key, {}))
                    return replace_refs(definition)
                else:
                    return { key: replace_refs(value) for key, value in obj.items() }
            elif isinstance(obj, list):
                return [ replace_refs(item) for item in obj ]
            else:
                return obj
        resolved_schema = replace_refs(json_schema)
        return resolved_schema